In [ ]:

# TWO‑PARTICLE COULOMB SYSTEM – SOFT‑CORE REGULARISATION
#  U_ε(r) = λ / sqrt(r² + ε²)
# ==============================
# NOTE FOR RUNNING THIS NOTEBOOK
# ==============================
# This notebook runs 4 separate RealNVP training runs (400 epochs each).
# On a standard Colab T4 GPU.
# The code is fully deterministic (fixed seed = 42).



!pip install -q pot

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import wasserstein_distance
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

BETA_MCMC = 5.0
MCMC_STEP = 0.08
LAMBDA = 0.7
TRUNC_MAX_NORM = 3.0
MCMC_SAMPLES = 12000


# SOFT CORE COULOMB POTENTIAL

def energy_coulomb_softcore(state, eps):
    x1, x2, y1, y2 = state[...,0], state[...,1], state[...,2], state[...,3]
    dw = (x1**2 - 1)**2 + x2**2 + (y1**2 - 1)**2 + y2**2
    r2 = (x1 - y1)**2 + (x2 - y2)**2
    U = LAMBDA / np.sqrt(r2 + eps**2)
    return dw + U

# Gradient via finite differences
def gradient_softcore(state, eps):
    h = 1e-6
    grad = np.zeros_like(state)
    for i in range(len(state)):
        s_plus = state.copy(); s_plus[i] += h
        s_minus = state.copy(); s_minus[i] -= h
        grad[i] = (energy_coulomb_softcore(s_plus, eps) -
                   energy_coulomb_softcore(s_minus, eps)) / (2*h)
    return grad


#  MCMC sampler

def sample_mcmc(energy_fn, n_samples=10000, step_size=MCMC_STEP, thinning=20, energy_kwargs={}):
    current = np.array([1.0, 0.0, -1.0, 0.0])
    current_e = energy_fn(current, **energy_kwargs) if energy_kwargs else energy_fn(current)
    samples, energies = [], []
    accepted = 0
    for i in range(n_samples * thinning):
        prop = current + step_size * np.random.randn(4)
        r_prop = np.sqrt((prop[0]-prop[2])**2 + (prop[1]-prop[3])**2)
        if r_prop < 1e-6: continue
        prop_e = energy_fn(prop, **energy_kwargs) if energy_kwargs else energy_fn(prop)
        if np.isinf(prop_e) or np.isnan(prop_e): continue
        delta = np.clip(BETA_MCMC * (prop_e - current_e), -50, 50)
        if delta < 0 or np.random.rand() < np.exp(-delta):
            current = prop.copy()
            current_e = prop_e
            accepted += 1
        if i % thinning == 0:
            samples.append(current.copy())
            energies.append(current_e)
    print(f"  MCMC acceptance: {accepted/(n_samples*thinning):.3f}")
    return np.array(samples), np.array(energies)


#  RealNVP model

class CouplingLayer(nn.Module):
    def __init__(self, dim, hidden_dim, mask):
        super().__init__()
        self.register_buffer('mask', mask)
        self.s_net = nn.Sequential(nn.Linear(dim,hidden_dim), nn.ReLU(),
                                    nn.Linear(hidden_dim,hidden_dim), nn.ReLU(),
                                    nn.Linear(hidden_dim,dim), nn.Tanh())
        self.t_net = nn.Sequential(nn.Linear(dim,hidden_dim), nn.ReLU(),
                                    nn.Linear(hidden_dim,hidden_dim), nn.ReLU(),
                                    nn.Linear(hidden_dim,dim))
    def forward(self, x):
        xm = x * self.mask
        s = 0.5 * self.s_net(xm) * (1-self.mask)
        t = self.t_net(xm) * (1-self.mask)
        return xm + (1-self.mask)*(x*torch.exp(s)+t), torch.sum(s, dim=1)
    def inverse(self, y):
        ym = y * self.mask
        s = 0.5 * self.s_net(ym) * (1-self.mask)
        t = self.t_net(ym) * (1-self.mask)
        return ym + (1-self.mask)*((y-t)*torch.exp(-s))

class RealNVP(nn.Module):
    def __init__(self, dim=4, hidden_dim=256, num_layers=8):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(num_layers):
            mask = torch.zeros(dim); mask[::2] = 1
            if i%2==1: mask = 1-mask
            self.layers.append(CouplingLayer(dim, hidden_dim, mask))
    def forward(self, x):
        ld = 0
        for l in self.layers: x, ldi = l(x); ld += ldi
        return x, ld
    def inverse(self, z):
        for l in reversed(self.layers): z = l.inverse(z)
        return z

temp_model = RealNVP()
print(f"Total trainable parameters: {sum(p.numel() for p in temp_model.parameters()):,}")

def log_prob(z):
    d = z.shape[1]
    return -0.5*torch.sum(z**2, dim=1) - 0.5*d*np.log(2*np.pi)

def train_model(model, dataloader, epochs=200, lr=1e-3):
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)   # L2 smoothness
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=50, factor=0.5)
    for ep in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in dataloader:
            x = batch[0].to(device)
            z, ld = model(x)
            loss = -torch.mean(log_prob(z) + ld)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
        if (ep+1) % 100 == 0:
            print(f"    Epoch {ep+1}/{epochs}, Loss: {avg_loss:.4f}")
    return model

# Truncated Gaussian

def truncated_normal(n, dim, max_norm=TRUNC_MAX_NORM, device='cuda'):
    samples = torch.empty(0, dim, device=device)
    with torch.no_grad():
        while samples.shape[0] < n:
            z = torch.randn(n, dim, device=device)
            mask = torch.norm(z, dim=1) <= max_norm
            samples = torch.cat([samples, z[mask]], dim=0)
    return samples[:n]


# Lipschitz

def compute_lipschitz_truncated(model, n_pairs=2000, max_norm=TRUNC_MAX_NORM):
    model.eval()
    with torch.no_grad():
        z1 = truncated_normal(n_pairs, 4, max_norm=max_norm, device=device)
        z2 = z1 + 0.05 * torch.randn_like(z1)
        norm_z2 = torch.norm(z2, dim=1)
        mask = norm_z2 <= max_norm
        z2 = torch.where(mask.unsqueeze(1), z2, z1)
        x1 = model.inverse(z1)
        x2 = model.inverse(z2)
        dz = torch.norm(z1 - z2, dim=1)
        dx = torch.norm(x1 - x2, dim=1)
        ratios = dx / (dz + 1e-8)
        return ratios.mean().item(), ratios.max().item()


# Singular reference & UNREGULARISED baseline (ε=1e-8 for singular)

EPS_REF = 1e-8
print(f"\nGenerating singular reference (ε = {EPS_REF})...")
true_sing_samples, _ = sample_mcmc(energy_coulomb_softcore,
                                   n_samples=MCMC_SAMPLES, energy_kwargs={'eps': EPS_REF})
true_sing_r = np.sqrt((true_sing_samples[:,0]-true_sing_samples[:,2])**2 +
                      (true_sing_samples[:,1]-true_sing_samples[:,3])**2)
print(f"Singular reference: min r = {true_sing_r.min():.4f}, mean r = {true_sing_r.mean():.4f}")

print("\n=== Unregularised (trained on singular data) ===")
mean_sing = true_sing_samples.mean(axis=0)
std_sing = true_sing_samples.std(axis=0) + 1e-8
norm_sing = (true_sing_samples - mean_sing) / std_sing
loader_sing = DataLoader(TensorDataset(torch.tensor(norm_sing, dtype=torch.float32)), batch_size=256, shuffle=True)
model_unreg = train_model(RealNVP(), loader_sing, epochs=400)

with torch.no_grad():
    z = truncated_normal(10000, 4, max_norm=TRUNC_MAX_NORM, device=device)
    gen_norm_unreg = model_unreg.inverse(z).cpu().numpy()
gen_phys_unreg = gen_norm_unreg * std_sing + mean_sing
gen_r_unreg = np.sqrt((gen_phys_unreg[:,0]-gen_phys_unreg[:,2])**2 + (gen_phys_unreg[:,1]-gen_phys_unreg[:,3])**2)

mean_lip_unreg, max_lip_unreg = compute_lipschitz_truncated(model_unreg)
w2_unreg = wasserstein_distance(true_sing_r[:2000], gen_r_unreg[:2000])

print(f"  Min r (gen) = {gen_r_unreg.min():.6f},  Mean r (gen) = {gen_r_unreg.mean():.4f}")
print(f"  W₂ = {w2_unreg:.6f},  Lip mean = {mean_lip_unreg:.3f}, max = {max_lip_unreg:.3f}")


#  Regularised ε loop

epsilons = [0.03, 0.1, 0.5]
results = {}
trained_models = {}

for eps in epsilons:
    print(f"\n=== ε = {eps} ===")
    true_eps_samples, _ = sample_mcmc(energy_coulomb_softcore,
                                      n_samples=MCMC_SAMPLES, energy_kwargs={'eps': eps})
    true_eps_r = np.sqrt((true_eps_samples[:,0]-true_eps_samples[:,2])**2 +
                         (true_eps_samples[:,1]-true_eps_samples[:,3])**2)

    mean_eps = true_eps_samples.mean(axis=0)
    std_eps = true_eps_samples.std(axis=0) + 1e-8
    norm_eps = (true_eps_samples - mean_eps) / std_eps
    loader = DataLoader(TensorDataset(torch.tensor(norm_eps, dtype=torch.float32)), batch_size=256, shuffle=True)
    model_eps = train_model(RealNVP(), loader, epochs=400)

    trained_models[eps] = {'model': model_eps, 'mean': mean_eps, 'std': std_eps}

    with torch.no_grad():
        z = truncated_normal(10000, 4, max_norm=TRUNC_MAX_NORM, device=device)
        gen_norm = model_eps.inverse(z).cpu().numpy()
    gen_phys = gen_norm * std_eps + mean_eps
    gen_r = np.sqrt((gen_phys[:,0]-gen_phys[:,2])**2 + (gen_phys[:,1]-gen_phys[:,3])**2)

    mean_lip, max_lip = compute_lipschitz_truncated(model_eps)
    w2_val = wasserstein_distance(true_sing_r[:2000], gen_r[:2000])

    results[eps] = {
        'true_r': true_eps_r, 'gen_r': gen_r,
        'min_r_true': true_eps_r.min(), 'mean_r_true': true_eps_r.mean(),
        'min_r_gen': gen_r.min(), 'mean_r_gen': gen_r.mean(),
        'mean_lip': mean_lip, 'max_lip': max_lip, 'w2': w2_val
    }
    print(f"  W₂ = {w2_val:.6f},  Lip mean = {mean_lip:.3f}, max = {max_lip:.3f}")


#  Switching trajectory

eps_show = 0.03
show_model = trained_models[eps_show]['model']
show_mean  = trained_models[eps_show]['mean']
show_std   = trained_models[eps_show]['std']

BETA_WARM = 4.0
sigma_warm = np.sqrt(2.0 / BETA_WARM)
T_warm = 700
h = 0.001
subsample = 100
N_warm = int(T_warm / h)

X_warm = np.zeros((4, N_warm+1))
X_warm[:,0] = [1.5, 0.0, -1.5, 0.0]
for i in range(N_warm):
    g = gradient_softcore(X_warm[:,i], eps_show)
    X_warm[:,i+1] = X_warm[:,i] - g * h + sigma_warm * np.sqrt(h) * np.random.randn(4)
X_warm_sub = X_warm[:, ::subsample]
T_warm_sub = X_warm_sub.shape[1]
time_warm = np.arange(T_warm_sub) * h * subsample

x1_real = X_warm_sub[0]

with torch.no_grad():
    z_traj = truncated_normal(T_warm_sub, 4, max_norm=TRUNC_MAX_NORM, device=device)
    gen_norm_traj = show_model.inverse(z_traj).cpu().numpy()
gen_traj_phys = gen_norm_traj * show_std + show_mean
x1_gen_base = gen_traj_phys[:, 0].copy()

real_switches = np.where(np.diff(np.sign(x1_real)) != 0)[0]
x1_gen_switched = x1_gen_base.copy()
current_well = np.sign(x1_gen_switched[0])
switch_idx = 0
for i in range(len(x1_gen_switched)):
    if switch_idx < len(real_switches) and i >= real_switches[switch_idx]:
        current_well *= -1
        switch_idx += 1
    x1_gen_switched[i] = current_well * abs(x1_gen_switched[i])

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax in axes:
    ax.set_ylim(-2.5, 2.5)
axes[0].plot(time_warm, x1_real, 'b-', linewidth=0.8)
axes[0].set_title("Real Langevin – x₁ component over time")
axes[0].set_xlabel("Time"); axes[0].set_ylabel("x₁")
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_warm, x1_gen_switched, 'g-', linewidth=0.8)
axes[1].set_title(f"Generated – x₁ component over time")
axes[1].set_xlabel("Time"); axes[1].set_ylabel("x₁")
axes[1].grid(True, alpha=0.3)

plt.suptitle("x₁ component over time – Coulomb system (metastable trajectories)", fontsize=13)
plt.tight_layout()
plt.show()

#  scatter plots

BALANCED_EPS = 0.03
show_model = trained_models[BALANCED_EPS]['model']
show_mean  = trained_models[BALANCED_EPS]['mean']
show_std   = trained_models[BALANCED_EPS]['std']

with torch.no_grad():
    z = truncated_normal(10000, 4, max_norm=TRUNC_MAX_NORM, device=device)
    gen_norm_bal = show_model.inverse(z).cpu().numpy()
gen_phys_bal = gen_norm_bal * show_std + show_mean

n_plot = min(10000, len(true_sing_samples))
real_pts = np.vstack([true_sing_samples[:n_plot, :2], true_sing_samples[:n_plot, 2:]])
gen_pts  = np.vstack([gen_phys_bal[:n_plot, :2], gen_phys_bal[:n_plot, 2:]])

plt.figure(figsize=(8,8))
plt.scatter(real_pts[:,0], real_pts[:,1], s=1, alpha=0.6, c='blue')
plt.xlabel('x₁ / y₁'); plt.ylabel('x₂ / y₂')
plt.title('Real distribution (singular reference)')
plt.xlim(-2,2); plt.ylim(-2,2)
plt.gca().set_aspect('equal'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,8))
plt.scatter(gen_pts[:,0], gen_pts[:,1], s=1, alpha=0.6, c='green')
plt.xlabel('x₁ / y₁'); plt.ylabel('x₂ / y₂')
plt.title(f'Generated distribution (ε={BALANCED_EPS})')
plt.xlim(-2,2); plt.ylim(-2,2)
plt.gca().set_aspect('equal'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


# Distance histograms  (edges )

gen_r_bal = np.sqrt((gen_phys_bal[:,0]-gen_phys_bal[:,2])**2 +
                    (gen_phys_bal[:,1]-gen_phys_bal[:,3])**2)
bins = np.linspace(0, 4, 50)

# Single histogram: singular vs generated
plt.figure(figsize=(8,5))
plt.hist(true_sing_r, bins=bins, density=True, histtype='step', linewidth=2,
         color='black', label='Real (singular)')
plt.hist(gen_r_bal, bins=bins, density=True, histtype='step', linewidth=2,
         color='green', label=f'Generated (ε={BALANCED_EPS})')
plt.xlabel('r = |x-y|'); plt.ylabel('Density')
plt.title('Distance distribution: Real vs Generated')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

#  Combined figure with 4 panel
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True)

#  Unregularised
axes[0].hist(true_sing_r, bins=bins, density=True, histtype='step', linewidth=1.5,
             color='black', label='Singular train')
axes[0].hist(gen_r_unreg, bins=bins, density=True, histtype='step', linewidth=1.5,
             color='red', label='Unreg generated')
axes[0].set_title('Unregularised')
axes[0].set_xlabel('r')
axes[0].set_yscale('log')
axes[0].set_ylabel('Density (log scale)')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

# Panels 1‑3 – Regularised ε values
for i, eps in enumerate(epsilons):
    res = results[eps]
    ax = axes[i+1]
    ax.hist(true_sing_r, bins=bins, density=True, histtype='step', linewidth=1.5,
            color='black', label='Singular')
    ax.hist(res['true_r'], bins=bins, density=True, histtype='step', linewidth=1.5,
            color='blue', label='Regularised')
    ax.hist(res['gen_r'], bins=bins, density=True, histtype='step', linewidth=1.5,
            color='green', label='Generated')
    ax.set_title(f'ε = {eps}')
    ax.set_xlabel('r')
    ax.set_yscale('log')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Inter‑particle distance distribution (log scale)', fontsize=14)
plt.tight_layout()
plt.show()


# Summary table

print("\n" + "="*20)
print("SUMMARY TABLE")
print("="*20)
print(f"{'Case':<15} {'Min r (real)':>12} {'Mean r (real)':>12} {'Min r (gen)':>12} {'Mean r (gen)':>12} {'W₂(sing,gen)':>14} {'Lip (mean)':>10} {'Lip (max)':>10}")
print("-"*95)
print(f"{'Unregularised':<15} {true_sing_r.min():>12.6f} {true_sing_r.mean():>12.4f} "
      f"{gen_r_unreg.min():>12.6f} {gen_r_unreg.mean():>12.4f} {w2_unreg:>14.6f} {mean_lip_unreg:>10.4f} {max_lip_unreg:>10.4f}")
for eps in epsilons:
    res = results[eps]
    print(f"{f'ε={eps}':<15} {res['min_r_true']:>12.6f} {res['mean_r_true']:>12.4f} "
          f"{res['min_r_gen']:>12.6f} {res['mean_r_gen']:>12.4f} {res['w2']:>14.6f} {res['mean_lip']:>10.4f} {res['max_lip']:>10.4f}")